<a href="https://colab.research.google.com/github/Zuhair0000/Fraud-Detiction/blob/main/fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import libraries

In [3]:
import polars as pl
import pandas as pd
import numpy as np

# Load Data

In [4]:
base_lf = pl.scan_csv("Base.csv")
print(base_lf.collect_schema())

Schema({'fraud_bool': Int64, 'income': Float64, 'name_email_similarity': Float64, 'prev_address_months_count': Int64, 'current_address_months_count': Int64, 'customer_age': Int64, 'days_since_request': Float64, 'intended_balcon_amount': Float64, 'payment_type': String, 'zip_count_4w': Int64, 'velocity_6h': Float64, 'velocity_24h': Float64, 'velocity_4w': Float64, 'bank_branch_count_8w': Int64, 'date_of_birth_distinct_emails_4w': Int64, 'employment_status': String, 'credit_risk_score': Int64, 'email_is_free': Int64, 'housing_status': String, 'phone_home_valid': Int64, 'phone_mobile_valid': Int64, 'bank_months_count': Int64, 'has_other_cards': Int64, 'proposed_credit_limit': Float64, 'foreign_request': Int64, 'source': String, 'session_length_in_minutes': Float64, 'device_os': String, 'keep_alive_session': Int64, 'device_distinct_emails_8w': Int64, 'device_fraud_count': Int64, 'month': Int64})


In [5]:
hidden_null_query = base_lf.select(
    (pl.col('prev_address_months_count') == -1).sum().alias('hidden_null_count')
)

print(hidden_null_query.collect())

shape: (1, 1)
┌───────────────────┐
│ hidden_null_count │
│ ---               │
│ u32               │
╞═══════════════════╡
│ 20594             │
└───────────────────┘


In [6]:
imbalance_query = base_lf.select(
    (pl.col('fraud_bool').mean() *100).alias('fraud_percentage')
)
print(imbalance_query.collect())

shape: (1, 1)
┌──────────────────┐
│ fraud_percentage │
│ ---              │
│ f64              │
╞══════════════════╡
│ 1.153794         │
└──────────────────┘


In [7]:
clean_lf = base_lf.with_columns(
    [
        pl.when(pl.col('prev_address_months_count') == -1)
        .then(1).otherwise(0).alias('is_missing_prev_address'),


        pl.when(pl.col('prev_address_months_count') == -1)
        .then(0).otherwise(pl.col('prev_address_months_count')).alias('prev_address_months_count')
    ]
)

clean_df = clean_lf.collect()

print(clean_df.select(["prev_address_months_count", "is_missing_prev_address"]).head(5))

shape: (5, 2)
┌───────────────────────────┬─────────────────────────┐
│ prev_address_months_count ┆ is_missing_prev_address │
│ ---                       ┆ ---                     │
│ i64                       ┆ i32                     │
╞═══════════════════════════╪═════════════════════════╡
│ 0                         ┆ 1                       │
│ 0                         ┆ 1                       │
│ 9                         ┆ 0                       │
│ 11                        ┆ 0                       │
│ 0                         ┆ 1                       │
└───────────────────────────┴─────────────────────────┘


# Train/Test Split

In [8]:
from sklearn.model_selection import train_test_split


sample_df = clean_lf.collect().sample(n=10000, seed=42)

pandas_df = sample_df.to_pandas()

y = pandas_df['fraud_bool']
X = pandas_df.drop(columns=['fraud_bool'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
print(y_train.shape)

(8000, 32)
(8000,)


# Data Preprocessing

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [10]:
cat_cols = ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']
num_cols = [col for col in X_train.columns if col not in cat_cols]

num_enc = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_enc = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


preprocessor = ColumnTransformer([
    ('num', num_enc, num_cols),
    ('cat', cat_enc, cat_cols)
])

# Machine Learning

In [11]:
rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

rf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['income',
                                                   'name_email_similarity',
                                                   'prev_address_months_count',
                                                   'current_address_months_count',
                                                   'customer_age',
                                                   'days_since_request',
                                                   'intended_balcon_amount',
                                                   'zip_count_4w',
                                                   'velocity_6h',
                                                   'velo...
                                                   'device_fraud_count',
                                                   'month',
                                                   'is_missing_prev_address']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['payment_type',
                                                   'employment_status',
                                                   'housing_status', 'source',
                                                   'device_os'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

# Deep Learning

In [12]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

In [14]:
print(f"Number of training batches: {len(train_loader)}")
print(f"Input feature shape: {X_train_tensor.shape[1]}")

Number of training batches: 125
Input feature shape: 53


In [15]:
import torch.nn as nn

class FraudNet(nn.Module):
  def __init__(self, input_shape):
    super(FraudNet, self).__init__()

    self.layer1 = nn.Linear(input_shape, 64)
    self.relu1 = nn.ReLU()
    self.dropout1 = nn.Dropout(p=0.3)

    self.layer2 = nn.Linear(64, 32)
    self.relu2 = nn.ReLU()
    self.dropout2 = nn.Dropout(p=0.3)

    self.layer3 = nn.Linear(32, 1)

  def forward(self, x):
    x = self.dropout1(self.relu1(self.layer1(x)))
    x = self.dropout2(self.relu2(self.layer2(x)))
    x = self.layer3(x)

    return x


model = FraudNet(input_shape=X_train_processed.shape[1])
print(model)

FraudNet(
  (layer1): Linear(in_features=53, out_features=64, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (layer3): Linear(in_features=32, out_features=1, bias=True)
)


In [16]:
import torch.optim as optim

num_negatives = (y_train_tensor == 0).sum()
num_positives = (y_train_tensor == 1).sum()
pos_weight_value = num_negatives / num_positives

print(f'Positive weight penalty: {pos_weight_value:.2f}')

Positive weight penalty: 87.89


In [17]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:
epochs = 10

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  for batch_X, batch_y in train_loader:
    optimizer.zero_grad()

    outputs = model(batch_X)

    loss = criterion(outputs, batch_y)

    loss.backward()

    optimizer.step()

    running_loss += loss.item()

  print(f"Epoch {epoch+1}/{epochs} - Average Loss: {running_loss/len(train_loader):.4f}")


Epoch 1/10 - Average Loss: 1.3316
Epoch 2/10 - Average Loss: 1.1297
Epoch 3/10 - Average Loss: 0.9651
Epoch 4/10 - Average Loss: 0.8885
Epoch 5/10 - Average Loss: 0.8484
Epoch 6/10 - Average Loss: 0.8251
Epoch 7/10 - Average Loss: 0.7441
Epoch 8/10 - Average Loss: 0.7495
Epoch 9/10 - Average Loss: 0.6743
Epoch 10/10 - Average Loss: 0.6643


In [19]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()

with torch.no_grad():
  test_outputs = model(X_test_tensor)
  probabilities = torch.sigmoid(test_outputs)
  predicitions = (probabilities > 0.5).float()

print("Model Evaluation")
print(classification_report(y_test_tensor, predicitions))

print("Confusion Matrix")
print(confusion_matrix(y_test_tensor, predicitions))

Model Evaluation
              precision    recall  f1-score   support

         0.0       1.00      0.82      0.90      1977
         1.0       0.04      0.65      0.08        23

    accuracy                           0.82      2000
   macro avg       0.52      0.74      0.49      2000
weighted avg       0.98      0.82      0.89      2000

Confusion Matrix
[[1618  359]
 [   8   15]]


In [20]:
from captum.attr import IntegratedGradients

ture_positives_mask = (y_test_tensor == 1) & (predicitions == 1)

tp_indices = torch.where(ture_positives_mask)[0]

fraudster_index = tp_indices[0]

fraudster_tensor = X_test_tensor[fraudster_index].unsqueeze(0)

ig = IntegratedGradients(model)

attributions, delta = ig.attribute(fraudster_tensor, return_convergence_delta=True)

attributions_np = attributions.squeeze(0).cpu().detach().numpy()

feature_importance = pd.DataFrame({
    "Feature_Index": range(len(attributions_np)),
    "Attribution_Score": attributions_np
})

top_fraud_features = feature_importance.sort_values(by="Attribution_Score", ascending=False).head(5)

print("\n Top 5 Reasons this transaction was flagged as FRAUD:")
print(top_fraud_features)


 Top 5 Reasons this transaction was flagged as FRAUD:
    Feature_Index  Attribution_Score
22             22           0.467291
29             29           0.388567
15             15           0.381665
51             51           0.360644
39             39           0.332275


In [23]:
feature_names = preprocessor.get_feature_names_out()

clean_feature_names = [name.split("__")[-1] for name in feature_names]

feature_importance_readable = pd.DataFrame({
    'Feature_Name': clean_feature_names,
    "Attribution_Score": attributions_np
})

top_fraud_reasons = feature_importance_readable.sort_values(by='Attribution_Score', ascending=False).head(5)
print(f"Transaction ID: {fraudster_index}")
print("Decision: FRAUD (Confidence > 50%)")
print("Key Driving Factors:")
for index, row in top_fraud_reasons.iterrows():
  print(f"- {row['Feature_Name']}, (Score: {row['Attribution_Score']:.4f})")

Transaction ID: 466
Decision: FRAUD (Confidence > 50%)
Key Driving Factors:
- keep_alive_session, (Score: 0.4673)
- payment_type_AC, (Score: 0.3886)
- phone_home_valid, (Score: 0.3817)
- device_os_windows, (Score: 0.3606)
- housing_status_BA, (Score: 0.3323)


In [24]:
import joblib
import torch

# Save the Scikit-Learn preprocessor
joblib.dump(preprocessor, 'fraud_preprocessor.joblib')

# Save the PyTorch model weights
torch.save(model.state_dict(), 'fraud_model_weights.pth')
print("Artifacts saved securely.")

Artifacts saved securely.
